# NLP-08 · Preference Learning and Alignment
### From demonstrations to chosen-versus-rejected behavior

SFT shows one desired response. Preference data compares two responses to the same
prompt. This lesson trains real DPO on the tiny SFT checkpoint, explains reward models,
and keeps PPO conceptual and correctly labeled. **Prerequisites:** NLP-07 and FND-02.


> **Canonical learner route · module NLP-08 of 67**
>
> Required prerequisites: **NLP-07, FND-02** · Previous: **NLP-07** ·
> Next after mastery: **EVAL-02** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives
- Build and audit `(prompt, chosen, rejected)` triples.
- Calculate Bradley–Terry and DPO losses.
- Sum response-token log probabilities conditioned on the prompt.
- Explain the frozen reference policy and beta.
- Distinguish KL-regularized RL from PPO's clipped surrogate.
- Measure preference gain and SFT retention without claiming safety.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · NLP-08 · Preference Learning and Alignment

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |


## 2 · Historical Motivation
RLHF commonly trains a preference reward model and optimizes a policy while limiting
drift. PPO is one possible optimizer. DPO derives an offline classification loss for a
specific KL-regularized preference model, removing online rollouts and a separately
deployed reward model but retaining strong data and evaluation assumptions.


## 3 · Intuition and Practical Problem
Two candidate answers receive a comparison, like a reviewer choosing the better draft.
The reference policy is an anchor: DPO asks whether the new policy improves the chosen
response *relative to how the reference scored both responses*. The analogy stops at
truth—reviewer preference can be inconsistent, biased, or wrong.

Use preference optimization only with meaningful comparisons after a strong SFT
baseline. Avoid it when supervised corrections suffice or preference labels lack
agreement and coverage.


## 4 · Mathematical Foundations
For reward scores $r_w,r_l$, Bradley–Terry loss is
$J_{RM}=-\log\sigma(r_w-r_l)$. $\sigma(z)=1/(1+e^{-z})$. If the margin is `1`, loss is
`-log(0.731)=0.313`; if it is `-1`, loss is `1.313`.

DPO uses policy $\pi_\theta$, frozen reference $\pi_{ref}$, chosen $y_w$, rejected
$y_l$, prompt $x$, and scale $\beta>0$:

$$J_{DPO}=-\log\sigma\{\beta[(\log\pi_\theta(y_w|x)-\log\pi_{ref}(y_w|x))
-(\log\pi_\theta(y_l|x)-\log\pi_{ref}(y_l|x))]\}.$$

Sequence log probability sums response-token log probabilities only. At zero relative
margin, loss is `log(2)=0.693`. For margin $m$, gradient magnitude is
$\sigma(-m)$: it approaches `1` for a very negative margin and `0` for a very positive
one. Length can confound summed log probabilities and must be audited.

The high-level RL objective $E[r]-\beta KL(\pi||\pi_{ref})$ is **not itself PPO**.
PPO additionally uses sampled rollouts, advantages, policy ratios, and clipping.


## 5 · Manual Implementation from Scratch
The project starts policy and reference from identical full-SFT weights, freezes the
reference, computes chosen/rejected response sequence log probabilities, and updates
only the policy with DPO.


In [ ]:
import sys
from pathlib import Path
repo=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'projects/language_model_adaptation').exists())
sys.path[:0]=[str(repo/'projects/language_model_adaptation/src'),str(repo/'projects/tiny_language_model/src')]
from language_model_adaptation.lab import run_adaptation_lab
report=run_adaptation_lab(seed=42)
print(report['preference_alignment'])


## 6 · Visualization


In [ ]:
import numpy as np, matplotlib.pyplot as plt
m=np.linspace(-5,5,200); loss=np.logaddexp(0,-m); grad=1/(1+np.exp(m))
plt.plot(m,loss,label='DPO logistic loss'); plt.plot(m,grad,label='gradient magnitude')
plt.xlabel('relative preference margin'); plt.legend(); plt.show()


## 7 · Failure Modes and Common Mistakes
| Symptom | Cause | Evidence | Repair |
|---|---|---|---|
| longer answers always lose | summed-length confound | length slice | balance/audit lengths |
| labels contradict | low agreement | annotator matrix | adjudicate or model uncertainty |
| preference rises, task falls | alignment tax | retention suite | tune beta/data/early stop |
| reward rises, behavior exploits gaps | reward hacking | adversarial human review | improve data/model and constrain rollout |

Never call preference accuracy truth, call the KL objective PPO, train the reference,
or include prompt tokens in response log probability.


## 8 · Library or Tool Implementation
DPO trainers can manage reference models, masking, padding, and distributed execution.
Verify chat-template boundaries, chosen/rejected lengths, reference identity, beta,
gradient accumulation, and evaluation code against the scratch loss before scaling.


## 9 · Realistic Case Study
A medical assistant prefers responses that acknowledge uncertainty and request source
verification. Clinicians label pairs with an abstain option. The team measures agreement,
factual correctness, safety, length, specialties, and SFT retention; preference alone
cannot authorize deployment.


## 10 · Learning and Production Considerations
Preference collection needs a rubric, randomized response order, annotator metadata,
privacy controls, disagreement handling, and held-out prompts. Monitor both the optimized
signal and independent outcomes that the signal might fail to represent.


## 11 · Tradeoff Analysis
| Method | Data | Strength | Weakness |
|---|---|---|---|
| SFT | target responses | simple baseline | no pairwise tradeoff signal |
| reward model | preference pairs | reusable score | exploitable proxy |
| PPO-style RLHF | reward + rollouts | online optimization | many moving parts |
| DPO | offline pairs | simpler executed loop | reference/data sensitivity |


## 12 · Readiness Check
The local DPO loss falls `0.693→near 0`, held-out preference accuracy changes `0→1`,
and SFT retention loss worsens. This proves the objective changed a narrow preference,
not that the model became aligned, safe, or truthful.


## 13 · Teach-Back
1. What is one preference example?
2. Why freeze the reference?
3. Which tokens form sequence log probability?
4. Why is negative-margin gradient large?
5. Why does preference accuracy not prove safety?


## 14 · Exercises, Self-Check, and Solutions
**Worked (10 min):** calculate logistic loss at margins `1` and `-1`: about `0.313`
and `1.313`. **Guided (20 min):** reverse one label; inspect its margin. **Independent
(45 min):** sweep beta and report preference plus retention. **Challenge (60 min):**
add annotator disagreement and a rule for ties/abstentions.

Summary: preference optimization changes relative response likelihood under a labeled
proxy and a reference constraint. **Memory aid:** *Optimize the comparison, anchor to
the reference, and audit everything the preference label leaves out.*


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **NLP-08 · Preference Learning and Alignment** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember NLP-08 · Preference Learning and Alignment: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · NLP-08

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
